In [ ]:
import os
if not os.path.exists("EduPredict"):
    !git clone https://github.com/oelaimar/EduPredict.git
os.chdir("EduPredict/notebooks")
!pip install -q -r ../requirements.txt

In [5]:
#import libraries and variables

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

X_train = pd.read_csv("../data/results/X_train.csv")
X_test = pd.read_csv("../data/results/X_test.csv")
Y_train = pd.read_csv("../data/results/Y_train.csv")["Exam_Score"]
Y_test = pd.read_csv("../data/results/Y_test.csv")["Exam_Score"]


In [3]:
# Linear Regression grid:

lr = LinearRegression()

lr_grid = {
    "fit_intercept" : [True, False],
    "positive" : [True, False]
}
lr_search = GridSearchCV(lr, lr_grid, cv=3, scoring="r2", n_jobs=-1)

lr_search.fit(X_train, Y_train)

best_lr = lr_search.best_estimator_

print("best params:", lr_search.best_params_)
print("Best CV R²:", lr_search.best_score_)



best params: {'fit_intercept': True, 'positive': False}
Best CV R²: 0.9898054550471498


##  SVR grid — same X_train/Y_train

In [4]:
srv = SVR()
svr_grid = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "epsilon" : [0.01, 0.1, 0.5, 1.0],
}
svr_search = GridSearchCV(srv, svr_grid, cv=3, scoring="r2", n_jobs=-1)
svr_search.fit(X_train, Y_train)

best_svr = svr_search.best_estimator_

print("SVR best params: ", svr_search.best_params_)
print("SVR best CV R²: ", svr_search.best_score_)

KeyboardInterrupt: 

## compare on the held-out test set (touch it only once):

In [ ]:
for name, search in [("LinearRegression", lr_search), ("SVR", svr_search)]:
    y_pred = search.predict(X_test)
    print(f"{name}:  R²={r2_score(Y_test, y_pred):.4f}  MAE={mean_absolute_error(Y_test, y_pred):.4f}")

## RandomForest grid

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_grid = {
    "n_estimators" : [100, 200, 300],
    "max_depth" : [None, 10, 20],
    "min_samples_split" : [2, 5, 10]
}

rf_search = GridSearchCV(
    rf,
    rf_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

rf_search.fit(X_train, Y_train)

best_rf = rf_search.best_estimator_

print("RF best params:", rf_search.best_params_)
print("RF best CV R²:", rf_search.best_score_)


## XGBoost grid — same X_train/Y_train

In [ ]:
xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1
)

xgb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 6, 10],
    "subsample": [0.8, 1.0]
}

xgb_search = GridSearchCV(
    xgb,
    xgb_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

xgb_search.fit(X_train, Y_train)

best_xgb = xgb_search.best_estimator_

print("XGB best params:", xgb_search.best_params_)
print("XGB best CV R²:", xgb_search.best_score_)


## one-time test comparison

In [ ]:

for name, search in [("RF", rf_search), ("XGB", xgb_search)]:
    y_pred = search.predict(X_test)

    print(
        f"{name}: "
        f"R²={r2_score(Y_test, y_pred):.4f}  "
        f"MAE={mean_absolute_error(Y_test, y_pred):.4f}"
    )

## evaluate best_estimator_ on the test set

In [ ]:
models = [
    ("LinearRegression", best_lr),
    ("SVR", best_svr),
    ("RF", best_rf),
    ("XGB", best_xgb)
]

for name, model in models:
    y_pred = model.predict(X_test)

    print(
        f"{name}: "
        f"R²={r2_score(Y_test, y_pred):.4f}  "
        f"MAE={mean_absolute_error(Y_test, y_pred):.4f}"
    )

## before/after table

In [6]:
baseline = pd.read_csv("../data/results/baseline.csv")

print(baseline)

models = [
    ("Linear Regression", best_lr),
    ("SVR", best_svr),
    ("Random Forest", best_rf),
    ("XGBoost", best_xgb)
]

after_results = []

for name, model in models:
    y_pred = model.predict(X_test)

    after_results.append({
        "Model": name,
        "After R²": r2_score(Y_test, y_pred),
        "After MAE": mean_absolute_error(Y_test, y_pred)
    })

after = pd.DataFrame(after_results)


               Model        R²       MAE      RMSE
0  Linear Regression  0.989717  0.271763  0.327151
1      Random Forest  0.891673  0.840500  1.061824
2            XGBoost  0.956514  0.531180  0.672759
3                SVR  0.971842  0.403659  0.541355
